# 08 · TabM: red neuronal tabular (corre en Kaggle con GPU)

TabM (Gorishniy et al., ICLR 2025) es un MLP que entrena en paralelo k = 32 submodelos que comparten la mayoría de
los pesos (BatchEnsemble) y promedia sus predicciones. Es una familia de modelos distinta a los árboles: aprende
fronteras suaves en vez de cortes, así que se espera que se equivoque en casos distintos y sume al ensamble.

Usa las mismas features y los mismos 5 folds que el resto (dataset privado `ap-tp3-features`).
Preprocesamiento: imputación por mediana + indicadores de faltantes, transformación por cuantiles a normal y
*piecewise linear embeddings* para las numéricas; las categóricas de baja cardinalidad entran como códigos.
Para cortar el entrenamiento (early stopping) se separa un 10% de la parte de entrenamiento de cada fold:
la parte de validación del fold nunca se usa para entrenar ni para elegir la época.

In [ ]:
!pip install -q tabm rtdl_num_embeddings

In [ ]:
import glob, os, time, warnings, numpy as np, pandas as pd, torch, tabm, rtdl_num_embeddings
warnings.filterwarnings("ignore")
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.preprocessing import QuantileTransformer

dev = "cuda" if torch.cuda.is_available() else "cpu"
print(dev, torch.cuda.get_device_name(0) if dev == "cuda" else "")
DEBUG = int(os.environ.get("DEBUG_ROWS", 0))            # solo para probar el codigo en local
hits = glob.glob("/kaggle/input/**/fold0_tr.parquet", recursive=True) or glob.glob("../kaggle_tabpfn/data/fold0_tr.parquet")
DATA = os.path.dirname(hits[0])
y = np.load(f"{DATA}/y.npy")
SEED, BATCH, LR, WD, MAX_EPOCHS, PATIENCE = 42, 256, 2e-3, 3e-4, (2 if DEBUG else 100), 8

In [ ]:
def prepare(A, B):
    """Ajusta el preprocesamiento con A y lo aplica a A y B. Devuelve tensores numericos, categoricos y cardinalidades."""
    cats = [c for c in A.columns if str(A[c].dtype) == "category"]
    nums = [c for c in A.columns if c not in cats]
    An, Bn = A[nums].astype("float32"), B[nums].astype("float32")
    miss = [c for c in nums if An[c].isna().any()]
    An = pd.concat([An, An[miss].isna().astype("float32").add_prefix("na_")], axis=1)
    Bn = pd.concat([Bn, Bn[miss].isna().astype("float32").add_prefix("na_")], axis=1)
    med = An.median()
    An, Bn = An.fillna(med), Bn.fillna(med)
    keep = An.columns[An.nunique() > 1]                     # columnas constantes no aportan (y rompen los bins)
    An, Bn = An[keep].values, Bn[keep].values
    rng = np.random.default_rng(SEED)
    noise = 1e-5 * An.std(0, keepdims=True) * rng.standard_normal(An.shape)   # desempata valores repetidos
    qt = QuantileTransformer(n_quantiles=1000, output_distribution="normal", subsample=10**9, random_state=SEED).fit(An + noise)
    An, Bn = qt.transform(An).astype("float32"), qt.transform(Bn).astype("float32")
    card, Ac, Bc = [], [], []
    for c in cats:
        dic = pd.Index(pd.concat([A[c].astype(str), B[c].astype(str)]).unique())
        card.append(len(dic)); Ac.append(dic.get_indexer(A[c].astype(str))); Bc.append(dic.get_indexer(B[c].astype(str)))
    t = lambda x, dt: torch.tensor(np.asarray(x), dtype=dt)
    return t(An, torch.float32), t(np.stack(Ac, 1), torch.long), t(Bn, torch.float32), t(np.stack(Bc, 1), torch.long), card

In [ ]:
@torch.no_grad()
def predict(model, Xn, Xc):
    model.eval()
    out = [torch.sigmoid(model(Xn[s:s + 4096].to(dev), Xc[s:s + 4096].to(dev)).squeeze(-1)).mean(1).cpu()
           for s in range(0, len(Xn), 4096)]
    return torch.cat(out).numpy()

def fit_predict(A, ya, B):
    torch.manual_seed(SEED); np.random.seed(SEED)
    tr, va = train_test_split(np.arange(len(A)), test_size=0.1, stratify=ya, random_state=SEED)  # para early stopping
    An, Ac, VBn, VBc, card = prepare(A.iloc[tr], pd.concat([A.iloc[va], B]))   # mismo preprocesamiento para val interna y B
    Vn, Vc, Bn, Bc = VBn[:len(va)], VBc[:len(va)], VBn[len(va):], VBc[len(va):]
    yt, yv = torch.tensor(ya[tr], dtype=torch.float32), ya[va]
    bins = rtdl_num_embeddings.compute_bins(An, n_bins=48)
    model = tabm.TabM.make(n_num_features=An.shape[1], cat_cardinalities=card, d_out=1,
                           num_embeddings=rtdl_num_embeddings.PiecewiseLinearEmbeddings(bins, 16, activation=False, version="B")).to(dev)
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WD)
    loss_fn = torch.nn.BCEWithLogitsLoss()
    best, best_state, bad = np.inf, None, 0
    for ep in range(MAX_EPOCHS):
        model.train()
        for idx in torch.randperm(len(An)).split(BATCH):
            logits = model(An[idx].to(dev), Ac[idx].to(dev)).squeeze(-1)            # (batch, k)
            loss = loss_fn(logits, yt[idx].to(dev)[:, None].expand_as(logits))     # cada submodelo con su propia perdida
            opt.zero_grad(); loss.backward(); opt.step()
        pv = np.clip(predict(model, Vn, Vc), 1e-6, 1 - 1e-6)
        ll = -np.mean(yv * np.log(pv) + (1 - yv) * np.log(1 - pv))
        if ll < best:
            best, bad, best_ep = ll, 0, ep
            best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
        else:
            bad += 1
            if bad >= PATIENCE:
                break
    model.load_state_dict(best_state)
    print(f"   mejor época {best_ep + 1} (logloss val interna {best:.4f})", flush=True)
    return predict(model, Bn, Bc)

In [ ]:
out = {"oof": np.zeros(len(y), dtype="float32")}
t0 = time.time()
for f, (a, b) in enumerate(StratifiedKFold(5, shuffle=True, random_state=0).split(y, y)):   # = features.cv_folds
    A, B = pd.read_parquet(f"{DATA}/fold{f}_tr.parquet"), pd.read_parquet(f"{DATA}/fold{f}_va.parquet")
    if DEBUG:
        A, B, a, b = A.iloc[:DEBUG], B.iloc[:DEBUG // 4], a[:DEBUG], b[:DEBUG // 4]
    p = fit_predict(A, y[a], B)
    out["oof"][b] = p; out[f"valid{f}"] = p
    print(f"fold {f}: accuracy {((p > .5) == y[b]).mean():.4f}  ({(time.time() - t0) / 60:.1f} min)", flush=True)
    np.savez("tabm.npz", **out)
    if DEBUG and f == 1:
        break
print(f"Accuracy CV TabM: {((out['oof'] > .5) == y).mean():.4f}")

In [ ]:
A, B = pd.read_parquet(f"{DATA}/full_tr.parquet"), pd.read_parquet(f"{DATA}/full_te.parquet")
if DEBUG:
    A, B = A.iloc[:DEBUG], B.iloc[:DEBUG // 4]
out["test"] = fit_predict(A, y[:len(A)], B)
np.savez("tabm.npz", **out)
print(f"test listo ({(time.time() - t0) / 60:.1f} min)")